# Totoro IELTS Master — Pipeline Tự Động Render Video Lip-Sync Giám Khảo Người Thật (0đ · Colab T4 GPU · MuseTalk v1.5)

Notebook này tương thích 100% với **Google Colab 2026 (Python 3.13 + CUDA 13.0 + T4 GPU)** bằng cách tự động dựng môi trường cô lập **Python 3.10 + PyTorch 2.0.1/cu118** qua `uv` (chỉ mất ~2 phút):
1. **Giọng Anh-Anh Neural chuẩn giới tính (Edge-TTS miễn phí):**
   - Ms. Claire: `en-GB-SoniaNeural`
   - Mr. Oliver: `en-GB-RyanNeural`
2. **240 câu hỏi cố định Part 1 & Part 3 + 9 câu dẫn chuẩn Cambridge** được render lip-sync khớp môi 100% bằng **MuseTalk v1.5** (`720p H.264 +faststart`, ~350–500 KB/clip).
3. Tự động xuất file **`manifest.json`** + **`totoro_r2_videos.zip`** sẵn sàng kéo thả lên **Cloudflare R2 (miễn phí 10 GB · 0đ băng thông)**.

> **Trước khi chạy:** Chọn menu `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# === BƯỚC 1: DỰNG MÔI TRƯỜNG PYTHON 3.10 + MUSETALK V1.5 (CHỐNG LỖI PYTHON 3.13) ===
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!apt-get -qq update && apt-get -qq install -y ffmpeg git curl unzip
!pip install -q uv edge-tts pillow tqdm

import os, subprocess
WORK = '/content/totoro_video_pipeline'
VENV = '/content/py310'
PY = f'{VENV}/bin/python'
PIP = f'{VENV}/bin/pip'
MIM = f'{VENV}/bin/mim'
os.makedirs(WORK, exist_ok=True)
%cd /content/totoro_video_pipeline

# 1) Tạo venv Python 3.10 sạch (tải binary mất ~5 giây)
if not os.path.exists(PY):
    !uv venv /content/py310 --python 3.10 --seed

# 2) Clone MuseTalk
if not os.path.exists('MuseTalk'):
    !git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git

# 3) Cài PyTorch 2.0.1 + cu118 + OpenMMLab + requirements trong /content/py310
check = subprocess.run([PY, '-c', 'import torch, mmpose, mmcv, mmdet, diffusers'], capture_output=True)
if check.returncode != 0:
    !uv pip install --python {PY} torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
    !uv pip install --python {PY} "numpy<2.0" "setuptools<70" "huggingface_hub==0.25.2" gdown openmim pyyaml matplotlib-inline
    !uv pip install --python {PY} -r MuseTalk/requirements.txt
    !{PIP} install --no-build-isolation chumpy
    !{MIM} install mmengine "mmcv==2.0.1" "mmdet==3.1.0" "mmpose==1.1.0"

# 4) Tải toàn bộ trọng số MuseTalk v1.5 (dùng thẳng huggingface.co, bỏ hf-mirror.com & pip install trong script gốc)
if not os.path.exists('MuseTalk/models/musetalkV15/unet.pth'):
    !uv pip install --python {PY} "huggingface_hub==0.25.2"
    %cd /content/totoro_video_pipeline/MuseTalk
    !sed -i 's/pip install/# pip install/g; s/gdown --id /gdown /g; s/export HF_ENDPOINT=/# export HF_ENDPOINT=/g' download_weights.sh
    !HF_ENDPOINT=https://huggingface.co PATH="/content/py310/bin:$PATH" sh download_weights.sh
    %cd /content/totoro_video_pipeline

!{PY} -c "import torch, mmpose, mmcv, mmdet; print('PY310 + MUSETALK V1.5 READY! Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())"
!ls -lh MuseTalk/models/musetalkV15/unet.pth

In [ ]:
# === BƯỚC 2: TẢI ẢNH GIÁM KHẢO & TRÍCH XUẤT ĐỦ 240 CÂU HỎI CAMBRIDGE + 9 CÂU LỆNH ===
import urllib.request, re, json, os
from PIL import Image

WORK = '/content/totoro_video_pipeline'
os.chdir(WORK)
REPO_RAW = 'https://raw.githubusercontent.com/ManTien-Dev/ielts-reading-vocab/main'
os.makedirs('inputs', exist_ok=True)
os.makedirs('audio', exist_ok=True)
os.makedirs('out_r2/claire/clips', exist_ok=True)
os.makedirs('out_r2/oliver/clips', exist_ok=True)

for name in ['claire', 'oliver']:
    url = f'{REPO_RAW}/assets/images/examiner_{name}_speak.webp'
    urllib.request.urlretrieve(url, f'inputs/{name}.webp')
    Image.open(f'inputs/{name}.webp').convert('RGB').resize((1280, 720)).save(f'inputs/{name}.png')
    if not os.path.exists(f'inputs/{name}_idle.mp4'):
        # Subtle breathing motion loop fallback (thay bằng clip Veo 3/Kling 5s nếu muốn chớp mắt thật)
        os.system(f"ffmpeg -y -loop 1 -i inputs/{name}.png -t 5 -vf \"scale=1312:738,crop=1280:720:16:9+3*sin(2*PI*t/5)\" -r 25 -c:v libx264 -pix_fmt yuv420p -movflags +faststart inputs/{name}_idle.mp4 -loglevel error")

# Trích xuất đủ 240 câu hỏi Part 1 & Part 3 (id dạng p1_c20t1_1, p3_c20t1_1...)
js_txt = urllib.request.urlopen(f'{REPO_RAW}/data/speaking_part13_data.js').read().decode('utf-8')
pairs = re.findall(r'"id"\s*:\s*"(p[13]_[^"]+)"\s*,\s*"q"\s*:\s*"([^"]+)"', js_txt)

# 9 câu lệnh chuẩn Cambridge IELTS
SYSTEM_CLIPS = [
    ('sys_p2_prep_intro', "Now, I'm going to give you a topic, and I'd like you to talk about it for one to two minutes. Before you talk, you'll have one minute to think about what you're going to say, and you can make some notes if you wish."),
    ('sys_p2_start_speak', "Alright? Remember you have one to two minutes for this, so don't worry if I stop you. I'll tell you when the time is up. Can you start speaking now, please?"),
    ('sys_p2_add_more', 'Would you like to add anything else?'),
    ('sys_p2_ready', 'Whenever you are ready, please begin talking about your topic.'),
    ('sys_probe_why', 'Why do you say that?'),
    ('sys_no_speech', 'Take your time. Would you like me to repeat the question?'),
    ('sys_end_p1', 'Thank you. That is the end of Part 1.'),
    ('sys_end_p2', 'Thank you. That is the end of Part 2.'),
    ('sys_end_p3', 'Thank you. That is the end of Part 3.'),
]

# Đặt 8 câu đầu (Walking + Fruit) + 9 câu hệ thống lên đầu để bản chạy thử test được ngay!
ALL_CLIPS = pairs[:8] + SYSTEM_CLIPS + pairs[8:]
print(f'Extracted {len(pairs)} Part 1/3 questions + {len(SYSTEM_CLIPS)} Cambridge system prompts = {len(ALL_CLIPS)} total clips per examiner!')
json.dump(ALL_CLIPS, open('clips_list.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

In [ ]:
# === BƯỚC 3: SINH ÂM THANH ANH-ANH NEURAL (NỮ SONIA / NAM RYAN) ===
import asyncio, edge_tts, json, os
from tqdm.notebook import tqdm

os.chdir('/content/totoro_video_pipeline')
ALL_CLIPS = json.load(open('clips_list.json', encoding='utf-8'))
VOICES = {
    'claire': 'en-GB-SoniaNeural',
    'oliver': 'en-GB-RyanNeural',
}

# Chạy thử nhanh 17 clip đầu (Topic Walking + Fruit + 9 câu Cambridge); khi chạy full đặt MAX_CLIPS = None
MAX_CLIPS = 17
TARGET_CLIPS = ALL_CLIPS[:MAX_CLIPS] if MAX_CLIPS else ALL_CLIPS

async def gen_all_tts():
    for ex_id, voice in VOICES.items():
        os.makedirs(f'audio/{ex_id}', exist_ok=True)
        for cid, text in tqdm(TARGET_CLIPS, desc=f'TTS {ex_id} ({voice})'):
            out_mp3 = f'audio/{ex_id}/{cid}.mp3'
            if os.path.exists(out_mp3) and os.path.getsize(out_mp3) > 1000:
                continue
            comm = edge_tts.Communicate(text, voice, rate='-4%')
            await comm.save(out_mp3)

await gen_all_tts()
print(f'British Neural TTS ready: {len(TARGET_CLIPS)} clips x 2 examiners!')

In [ ]:
# === BƯỚC 4: CHẠY MUSETALK V1.5 BATCH LIP-SYNC & NÉN 720P FASTSTART ===
import yaml, os, json, subprocess

os.chdir('/content/totoro_video_pipeline')
PY = '/content/py310/bin/python'
ALL_CLIPS = json.load(open('clips_list.json', encoding='utf-8'))
TARGET_CLIPS = ALL_CLIPS[:MAX_CLIPS] if MAX_CLIPS else ALL_CLIPS

for ex_id in ['claire', 'oliver']:
    cfg = {}
    # Ưu tiên clip {ex_id}_speak.mp4 (có cử chỉ tay & chớp mắt từ Veo 3/Kling), nếu chưa có thì fallback về idle.mp4
    speak_vid = f'/content/totoro_video_pipeline/inputs/{ex_id}_speak.mp4'
    idle_vid = f'/content/totoro_video_pipeline/inputs/{ex_id}_idle.mp4'
    src_video = speak_vid if os.path.exists(speak_vid) else idle_vid
    for cid, _ in TARGET_CLIPS:
        dst_mp4 = f'/content/totoro_video_pipeline/out_r2/{ex_id}/clips/{cid}.mp4'
        if os.path.exists(dst_mp4) and os.path.getsize(dst_mp4) > 20000:
            continue
        cfg[f'{ex_id}_{cid}'] = {
            'video_path': src_video,
            'audio_path': f'/content/totoro_video_pipeline/audio/{ex_id}/{cid}.mp3',
        }
    if cfg:
        yaml_path = f'/content/totoro_video_pipeline/MuseTalk/configs/inference/totoro_{ex_id}.yaml'
        with open(yaml_path, 'w') as f:
            yaml.safe_dump(cfg, f)
        print(f'Running MuseTalk v1.5 batch for {ex_id} ({len(cfg)} clips)...')
        %cd /content/totoro_video_pipeline/MuseTalk
        !{PY} -m scripts.inference --inference_config {yaml_path} --result_dir /content/totoro_video_pipeline/raw_out/{ex_id} --unet_config models/musetalkV15/musetalk.json --unet_model_path models/musetalkV15/unet.pth --version v15 --ffmpeg_path /usr/bin/ffmpeg --batch_size 8
        %cd /content/totoro_video_pipeline

    # Tìm đệ quy file mp4 trong raw_out/{ex_id} (kể cả thư mục con v15/) và nén nhẹ web-ready
    raw_dir = f'/content/totoro_video_pipeline/raw_out/{ex_id}'
    all_raw = []
    if os.path.exists(raw_dir):
        for root, _, files in os.walk(raw_dir):
            for fn in files:
                if fn.endswith('.mp4'):
                    all_raw.append(os.path.join(root, fn))

    done_cnt = 0
    for cid, _ in TARGET_CLIPS:
        dst_mp4 = f'/content/totoro_video_pipeline/out_r2/{ex_id}/clips/{cid}.mp4'
        if os.path.exists(dst_mp4) and os.path.getsize(dst_mp4) > 20000:
            done_cnt += 1
            continue
        matches = [p for p in all_raw if cid in os.path.basename(p)]
        if matches:
            src_mp4 = matches[0]
            os.system(f'ffmpeg -y -i "{src_mp4}" -c:v libx264 -crf 24 -preset veryfast -pix_fmt yuv420p -c:a aac -b:a 96k -movflags +faststart "{dst_mp4}" -loglevel error')
            if os.path.exists(dst_mp4) and os.path.getsize(dst_mp4) > 20000:
                done_cnt += 1
    print(f'==> {ex_id}: {done_cnt}/{len(TARGET_CLIPS)} lip-sync MP4 clips ready!')

print('Batch Lip-Sync v1.5 complete!')

In [ ]:
# === BƯỚC 5: XUẤT MANIFEST.JSON & ĐÓNG GÓI TOTORO_R2_VIDEOS.ZIP ===
import os, json, shutil

os.chdir('/content/totoro_video_pipeline')
# Thay R2_PUBLIC_URL bằng Public Bucket URL của Cloudflare R2 (VD: https://pub-xxxx.r2.dev/)
R2_PUBLIC_URL = 'https://YOUR-R2-BUCKET.r2.dev/'

manifest = {
    'version': 2,
    'baseUrl': R2_PUBLIC_URL,
    'simliAutoConnect': True,
    'examiners': {}
}

for ex_id in ['claire', 'oliver']:
    base_loops = {}
    for st in ['idle', 'speak', 'nod', 'notes']:
        p = f'inputs/{ex_id}_{st}.mp4'
        if os.path.exists(p):
            dst = f'out_r2/{ex_id}/{st}.mp4'
            shutil.copy2(p, dst)
            base_loops[st] = f'{ex_id}/{st}.mp4'
        else:
            base_loops[st] = ''
    clips_dir = f'out_r2/{ex_id}/clips'
    clips = {}
    for fn in sorted(os.listdir(clips_dir)):
        if fn.endswith('.mp4'):
            cid = fn[:-4]
            clips[cid] = f'{ex_id}/clips/{fn}'
    manifest['examiners'][ex_id] = {
        'baseLoops': base_loops,
        'clips': clips
    }
    print(f'{ex_id}: {len(clips)} clips in manifest')

json.dump(manifest, open('out_r2/manifest.json', 'w', encoding='utf-8'), indent=2)
!rm -f /content/totoro_r2_videos.zip && zip -rq /content/totoro_r2_videos.zip out_r2
!ls -lh /content/totoro_r2_videos.zip